# How to evaluate correspondences?

`EvaluationManager` provides a unified way to score any correspondence method.
It mirrors the `LossManager` pattern from training: each metric declares the
inputs it needs via `required_inputs`, and metrics are **silently skipped**
when those inputs are absent — so the same evaluator works for full-shape,
partial-shape, and soft-correspondence methods alike.

In [1]:
import numpy as np

from geomfum.dataset import NotebooksDataset
from geomfum.evaluation import (
    CorrespondenceMetric,
    CoverageCountMetric,
    CoverageMetric,
    DirichletEnergyMetric,
    EuclideanErrorMetric,
    EvaluationManager,
    GeodesicErrorMetric,
    OverlapIoUMetric,
)
from geomfum.matcher import FunctionalMapMatcher
from geomfum.metric import HeatDistanceMetric
from geomfum.shape import TriangleMesh

[Load meshes](00_load_mesh_from_file.ipynb).

In [2]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("faust-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("faust-04"))

mesh_a.n_vertices, mesh_b.n_vertices

INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-00.off').
INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-04.off').


(6890, 6890)

## Compute a correspondence

[Use a matcher](19_matchers.ipynb) to get a `CorrespondenceResult`.

In [3]:
matcher = FunctionalMapMatcher()
result = matcher(mesh_a, mesh_b)

print(f"p2p21 shape : {result.p2p21.shape}")
print(f"fmap12 shape: {result.fmap12.shape}")


Requested k=200 is larger than the number of eigenvalues currently in use (spectrum_size=200). Recomputing basis with k=200.
Requested k=200 is larger than the number of eigenvalues currently in use (spectrum_size=200). Recomputing basis with k=200.
p2p21 shape : (6890,)
fmap12 shape: (30, 30)


## Ground truth correspondences

FAUST meshes share the same topology and vertex ordering, so the ground truth
is the identity map: vertex $i$ in B corresponds to vertex $i$ in A.

In [4]:
n = mesh_a.n_vertices
corr_a = np.arange(n)  # ground-truth indices in A
corr_b = np.arange(n)  # ground-truth indices in B

## Geodesic distance matrix

Geodesic error requires a precomputed distance matrix on the target shape.
This can take a moment; in practice it is computed once and cached per shape.

In [5]:
dist_metric = HeatDistanceMetric.from_registry(mesh_a, which="pp3d")
dist_a = dist_metric.dist_matrix()  # shape [n_vertices_a, n_vertices_a]

dist_a.shape

(6890, 6890)

## Using a single metric

Each metric is a callable that receives a flat inputs dict.
Call it directly for a quick one-off measurement.

In [6]:
geo_metric = GeodesicErrorMetric()

inputs = result.to_dict()  # fmap12, p2p21, …
inputs["dist_a"] = dist_a
inputs["corr_a"] = corr_a
inputs["corr_b"] = corr_b

geo_metric(inputs)

0.02509681450395355

## EvaluationManager

`EvaluationManager` assembles the inputs dict for you and runs every
registered metric whose `required_inputs` are satisfied.

In [7]:
evaluator = EvaluationManager(
    [
        GeodesicErrorMetric(),
        EuclideanErrorMetric(),
        DirichletEnergyMetric(),
        CoverageMetric(),
        CoverageCountMetric(),
        OverlapIoUMetric(),  # needs mask_a — will be skipped
    ]
)

scores = evaluator.compute(
    result,
    shape_a=mesh_a,
    shape_b=mesh_b,
    corr_a=corr_a,
    corr_b=corr_b,
    dist_a=dist_a,
)

for name, value in scores.items():
    print(f"{name:<30} {value:.4f}")

GeodesicErrorMetric            0.0251
EuclideanErrorMetric           0.0234
DirichletEnergyMetric          0.0013
CoverageMetric                 0.6156
CoverageCountMetric            0.4916


`OverlapIoUMetric` requires `mask_a` (a partial-shape concept), so it was
skipped automatically — no errors, no special casing needed.

## Metrics are skipped when inputs are missing

Omitting `dist_a` drops all geodesic metrics; the rest still run.

In [8]:
scores_no_dist = evaluator.compute(
    result,
    shape_a=mesh_a,
    shape_b=mesh_b,
    corr_a=corr_a,
    corr_b=corr_b,
    # dist_a omitted
)

print("Available:", list(scores_no_dist.keys()))

Available: ['EuclideanErrorMetric', 'DirichletEnergyMetric', 'CoverageMetric', 'CoverageCountMetric']


## Named metrics

Pass a `dict` instead of a list to control the output key names.

In [9]:
evaluator_named = EvaluationManager(
    {
        "geo_err": GeodesicErrorMetric(),
        "coverage": CoverageMetric(),
    }
)

evaluator_named.compute(
    result, shape_a=mesh_a, dist_a=dist_a, corr_a=corr_a, corr_b=corr_b
)

{'geo_err': 0.02509681450395355, 'coverage': 0.6156014784589505}

## Plain dict input

`EvaluationManager` also accepts any plain `dict`, so it works with outputs
from methods that do not return a `CorrespondenceResult`.

In [11]:
plain_output = {"p2p21": result.p2p21}

evaluator.compute(
    plain_output,
    shape_a=mesh_a,
    shape_b=mesh_b,
    corr_a=corr_a,
    corr_b=corr_b,
    dist_a=dist_a,
)

{'GeodesicErrorMetric': 0.02509681450395355,
 'EuclideanErrorMetric': 0.023372071443387595,
 'DirichletEnergyMetric': 0.0012922603504430836,
 'CoverageMetric': 0.6156014784589505,
 'CoverageCountMetric': 0.4915820029027576}

## Further reading

* [How to compute correspondences with matchers?](19_matchers.ipynb)
* [How to visualize distances on a mesh?](17_vis_dist.ipynb)
* [How to use deep functional map matchers?](20_deep_fm_matcher.ipynb)